
# Defacing

Simple example.

Example on how to run the defacing pre-processing using BrainPrep.
See `user guide <defacing>` for details.

## Data

Let's first get some anatomical data: T1w, T2w and FLAIR..


In [ ]:
from pathlib import Path
from brainprep.utils import Bunch
from brainprep.datasets import OpenMSDataset

datadir = Path("/tmp/brainprep-data")
datadir.mkdir(parents=True, exist_ok=True)
dataset = OpenMSDataset(datadir)
data = Bunch()
for modality in ("T1w", "T2w", "FLAIR"):
    data[modality] = Bunch(
        sub01=dataset.fetch(
            subject="01",
            modality=modality,
            dtype="cross_sectional",
        ),
        sub02=dataset.fetch(
            subject="02",
            modality=modality,
            dtype="cross_sectional",
        ),
    )
print(data)

## Analysis

Let's now perform preprocessing using the BrainPrep.
As with many tutorials, we won't execute the code directly here.
However, feel free to set the 'dryrun' configuration to False
to actually run each step and generate results on disk.



In [ ]:
import shutil
from brainprep.workflow import (
    brainprep_defacing,
    brainprep_group_defacing,
)
from brainprep.config import Config

outdir = Path("/tmp/brainprep-defacing")
if outdir.is_dir():
    shutil.rmtree(outdir)
outdir.mkdir(parents=True, exist_ok=True)
with Config(dryrun=True, verbose=True):
    for modality, modality_data in data.items():
        for subject_data in modality_data.values():
            outputs = brainprep_defacing(
                anatomical_file=subject_data.anat,
                output_dir=outdir,
                keep_intermediate=True,
            )
            outputs.deface_anatomical_file.touch(exist_ok=True)
            outputs.mask_file.touch(exist_ok=True)
        outputs = brainprep_group_defacing(
            modality=modality,
            output_dir=outdir,
        )

## CLI

Let's now generate the same analysis using the CLI. The goal here is to
translate the workflow calls into explicit shell commands.
See `user guide <cli>` for details.



In [ ]:
from pprint import pprint

commands = []
commands.append(
    [
        [
            "brainprep", "subject-level-defacing",
            "--anatomical_file", str(subject_data.anat),
            "--output-dir", str(outdir),
            "--keep-intermediate",
        ]
        for subject_data in data["T1w"].values()
    ]
)
commands.append(
    [
        [
            "brainprep", "subject-level-defacing",
            "--anatomical_file", str(subject_data.anat),
            "--output-dir", str(outdir),
            "--keep-intermediate",
        ]
        for mod in ("T2w", "FLAIR")
        for subject_data in data[mod].values()
    ]
)
commands.append(
    [
        [
            "brainprep", "group-level-defacing",
            "--modality", modality,
            "--output-dir", str(outdir),
        ]
        for modality in data.keys()
    ]
)
pprint(commands)

## Container

Note that the commands generated by the CLI are not limited to being
displayed for reference; they can also be executed directly within the
workflow‑dedicated container. By running the commands inside the container,
you benefit from a controlled runtime context where all necessary
dependencies, libraries, and configuration files are already available.
In practice, this means that once the CLI has produced the appropriate
instructions, you can simply copy and run them inside the container to
achieve the intended results. You can find the BrainPrep images on Docker
Hub: [Neurospin Docker Hub](https://hub.docker.com/u/neurospin).

